In [1]:
# PYTHON: Import the built-in library for handling Comma Separated Values.
# SOC: Logs are often exported from a SIEM (Security Information and Event Management) in CSV format. We need a tool to read this structured evidence.
import csv

# PYTHON: Store the name of our target file in a variable.
# SOC: This is our raw evidence file containing firewall network traffic and server authentication logs.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP OUR METRICS (BASELINE)
# ==========================================
# PYTHON: Initialize an integer variable to 0.
# SOC: We need to know the total volume of traffic. If we normally see 1,000 events a day, and today we see 100,000, that spike is an anomaly we must investigate.
total_events = 0

# PYTHON: Initialize counters for specific actions.
# SOC: We categorize traffic to understand the network's "normal" state.
success_events = 0  # SOC: Legitimate traffic that was allowed through.
failed_events = 0   # SOC: Traffic that failed (e.g., bad password). High numbers here indicate a problem.
blocked_events = 0  # SOC: Traffic the firewall successfully stopped.

# ==========================================
# PHASE 2: SETTING UP OUR THREAT BUCKETS
# ==========================================
# PYTHON: Create an empty list [] to store dictionary objects later.
# SOC: We will store every single failed login event here for further review.
failed_logins_list = []

# PYTHON: Create an empty dictionary {} for key-value mapping.
# SOC: We will use this to map an IP address (Key) to the number of times it failed to log in (Value). This tracks repeat offenders.
brute_force_counts = {}

# PYTHON: Create an empty list for severe events.
# SOC: This is for Alert Triage. We need a bucket specifically for "hair-on-fire" alerts so they don't get lost in the noise.
high_critical_events = []

# PYTHON: Create an empty list for geolocation anomalies.
# SOC: If a user logs in from New York, and 5 minutes later logs in from China, that is physically impossible. We track foreign logins to catch compromised accounts.
suspicious_foreign_logins = []


# ==========================================
# PHASE 3: PARSING THE EVIDENCE
# ==========================================
# PYTHON: Open the file in 'r' (read) mode. 'with' ensures the file closes automatically when done.
# SOC: Chain of Custody. We strictly READ the evidence. We never write to or alter the original log file.
with open(file_path, mode='r') as csvfile:
    
    # PYTHON: Use DictReader. It reads the top row of the CSV and uses those names as keys for the data below.
    # SOC: This standardizes our data. . We can now call exact fields like 'source_ip'.
    reader = csv.DictReader(csvfile)
    
    # PYTHON: Start a loop. Look at one row, run the code below, then move to the next row.
    # SOC: This is our automated ingestion. The script will analyze every single packet of data recorded by the firewall.
    for row in reader:
        
        # PYTHON: Add 1 to our total_events counter.
        # SOC: Tallying the total log volume for our baseline metric.
        total_events += 1
        
        # --------------------------------------
        # TASK 2: CATEGORIZE THE ACTION
        # --------------------------------------
        # PYTHON: Extract the 'action' column value and make it lowercase to avoid case-sensitivity errors (e.g., 'Failed' vs 'failed').
        # SOC: What was the result of the event? Did the firewall allow it or drop it?
        action = row['action'].lower()
        
        # PYTHON: If the action was 'success'...
        if action == 'success':
            # PYTHON: ...add 1 to the success counter.
            success_events += 1
            
        # PYTHON: Else, if the action was 'failed'...
        elif action == 'failed':
            # PYTHON: ...add 1 to the failed counter.
            failed_events += 1
            
        # PYTHON: Else, if the action was 'blocked' OR 'detected'...
        elif action == 'blocked' or action == 'detected': 
            # PYTHON: ...add 1 to the blocked counter.
            # SOC: A 'blocked' event means our defenses worked! But we still track them to see who is trying to attack us.
            blocked_events += 1
            
        # --------------------------------------
        # TASK 3: FAILED LOGIN DETECTION
        # --------------------------------------
        # PYTHON: Extract the 'event_type' column and make it lowercase.
        # SOC: Was this a file download, a port scan, or a login attempt?
        event_type = row['event_type'].lower()
        
        # PYTHON: Check if TWO conditions are met: the event is a 'login' AND the action was 'failed'.
        # SOC: We are filtering out successful logins and non-login failures (like failing to download a file). We only care about bad passwords right now.
        if event_type == 'login' and action == 'failed':
            
            # PYTHON: Add a new dictionary containing specific details to our failed_logins list.
            # SOC: We don't need the whole log. We extract the "Who" (username), "Where from" (source_ip, country), and "When" (timestamp) to build a profile of the failure.
            failed_logins_list.append({
                'timestamp': row['timestamp'],
                'source_ip': row['source_ip'],
                'username': row['username'],
                'country': row['country']
            })
            
            # --------------------------------------
            # TASK 4: BRUTE FORCE DETECTION
            # --------------------------------------
            # PYTHON: Extract the source IP address of the attacker.
            # SOC: The Source IP is the digital fingerprint of the computer attacking us.
            ip = row['source_ip']
            
            # PYTHON: Update the dictionary. Get the current count for this IP (default to 0 if it's new), then add 1.
            # SOC: [Image illustrating a brute force password attack where one computer rapid-fires passwords]. A single failed login is usually a typo. Multiple failed logins from the SAME IP is a script trying thousands of passwords. We are building a tracker.
            brute_force_counts[ip] = brute_force_counts.get(ip, 0) + 1
            
        # --------------------------------------
        # TASK 5: TRIAGE HIGH/CRITICAL ALERTS
        # --------------------------------------
        # PYTHON: Extract the severity level from the log.
        # SOC: Firewalls assign severity. 'Low' might be a normal login. 'Critical' might be Malware detected.
        severity = row['severity'].lower()
        
        # PYTHON: Check if the severity is in the list of ['high', 'critical'].
        # SOC: Alert Fatigue is real. SOC analysts get thousands of alerts. We use this to isolate the most dangerous events so they are investigated first.
        if severity in ['high', 'critical']:
            
            # PYTHON: Add the relevant details to our high/critical list.
            # SOC: Documenting the severe event for the final report.
            high_critical_events.append({
                'timestamp': row['timestamp'],
                'username': row['username'],
                'event_type': row['event_type'],
                'severity': row['severity']
            })
            
        # --------------------------------------
        # TASK 6: SUSPICIOUS GEOLOCATION
        # --------------------------------------
        # PYTHON: Check if it's a login AND the country is NOT 'US'.
        # SOC: This is called "geo-fencing." If our entire workforce is in the United States, a login from another country is highly suspicious and requires immediate account lockdown.
        if event_type == 'login' and row['country'] != 'US':
            
            # PYTHON: Add the details to our foreign logins list.
            # SOC: We include the 'action' (success/fail). A failed foreign login is a scan. A SUCCESSFUL foreign login means the attacker is inside the network.
            suspicious_foreign_logins.append({
                'source_ip': row['source_ip'],
                'username': row['username'],
                'country': row['country'],
                'action': row['action']
            })

# ==========================================
# PHASE 4: GENERATING THE REPORT
# ==========================================
# PYTHON: Print a header to the console.
# SOC: Formatting the output so it's readable for the senior security team or management.
print("--- Task 3: Failed Logins ---")

# PYTHON: Loop through the list of failed logins we built earlier.
# SOC: Presenting the raw data of the failed authentication attempts.
for entry in failed_logins_list:
    # PYTHON: Print a formatted string (f-string) injecting the variables.
    print(f"Timestamp: {entry['timestamp']}, IP: {entry['source_ip']}, User: {entry['username']}, Country: {entry['country']}")

# PYTHON: Print the Brute Force header with a newline (\n) for spacing.
print("\n--- Task 4: Brute Force Detection ---")

# PYTHON: Create an empty list to keep track of the specific IPs that crossed the threshold.
brute_force_ips = []

# PYTHON: Loop through the dictionary, unpacking both the IP (key) and the Count (value).
for ip, count in brute_force_counts.items():
    
    # PYTHON: Check if the count is greater than or equal to 3.
    # SOC: This is our detection threshold. Less than 3 is a human mistake. 3 or more is an automated attack.
    if count >= 3:
        
        # PYTHON: Print the warning to the screen.
        # SOC: Generating an Actionable Alert. The analyst will now take this IP and block it on the firewall.
        print(f"Brute Force Attempt: {ip} - Attempts: {count}")
        
        # PYTHON: Save the bad IP to our list so we can count total bad IPs later.
        brute_force_ips.append(ip)

# PYTHON: Print the High/Critical header.
print("\n--- Task 5: High/Critical Severity ---")

# PYTHON: Loop through the high/critical list and print details.
# SOC: This section forms the "To-Do" list for the Incident Response team.
for entry in high_critical_events:
    print(f"Timestamp: {entry['timestamp']}, User: {entry['username']}, Event: {entry['event_type']}, Severity: {entry['severity']}")

# PYTHON: Print the Foreign Logins header.
print("\n--- Task 6: Suspicious Foreign Logins ---")

# PYTHON: Loop through and print.
# SOC: Highlighting potential compromised accounts based on location data.
for entry in suspicious_foreign_logins:
    print(f"IP: {entry['source_ip']}, User: {entry['username']}, Country: {entry['country']}, Action: {entry['action']}")

# --------------------------------------
# TASK 7: THE EXECUTIVE SUMMARY
# --------------------------------------
# PYTHON: Print the final summary dashboard.
# SOC: Management doesn't want to see 10,000 lines of logs. They want to see the bottom line. How many attacks happened, and how many severe events need funding to fix?
print("\n========== SECURITY REPORT ==========")

# PYTHON: Print the variable counters and the length (len()) of our lists.
print(f"Total Events: {total_events}")
print(f"Failed Logins: {len(failed_logins_list)}")
print(f"Brute Force IPs: {len(brute_force_ips)}")
print(f"High/Critical Events: {len(high_critical_events)}")
print(f"Suspicious Foreign Logins: {len(suspicious_foreign_logins)}")
print("=====================================")

--- Task 3: Failed Logins ---
Timestamp: 2024-02-03 08:18:12, IP: 203.0.113.45, User: admin, Country: CN
Timestamp: 2024-02-03 08:18:34, IP: 203.0.113.45, User: admin, Country: CN
Timestamp: 2024-02-03 08:18:56, IP: 203.0.113.45, User: admin, Country: CN
Timestamp: 2024-02-03 08:19:18, IP: 203.0.113.45, User: root, Country: CN
Timestamp: 2024-02-03 08:19:42, IP: 203.0.113.45, User: root, Country: CN
Timestamp: 2024-02-03 08:20:05, IP: 203.0.113.45, User: root, Country: CN
Timestamp: 2024-02-03 08:20:28, IP: 203.0.113.45, User: root, Country: CN
Timestamp: 2024-02-03 08:45:33, IP: 185.220.101.33, User: admin, Country: TOR
Timestamp: 2024-02-03 08:46:01, IP: 185.220.101.33, User: admin, Country: TOR
Timestamp: 2024-02-03 09:20:33, IP: 203.0.113.45, User: admin, Country: CN
Timestamp: 2024-02-03 10:00:15, IP: 203.0.113.45, User: admin, Country: CN
Timestamp: 2024-02-03 10:35:30, IP: 185.220.101.33, User: root, Country: TOR
Timestamp: 2024-02-03 12:10:00, IP: 203.0.113.45, User: jsmith, Co

In [2]:
# PYTHON: Import the CSV module to read our structured log data.
import csv

# PYTHON: Define the name of our input file (the raw logs).
input_file = 'Siem_analysis_enterprise.csv'

# PYTHON: Define the name of our output file.
# SOC: We name this clearly so the Incident Response (IR) team knows exactly what it is.
output_file = 'SOC_Investigation_Report.txt'

# ==========================================
# PHASE 1 & 2: SETTING UP METRICS & BUCKETS
# ==========================================
# PYTHON: Set up our counters for the baseline traffic analysis.
total_events = 0
success_events = 0
failed_events = 0
blocked_events = 0

# PYTHON: Set up our lists and dictionaries to hold specific threat data.
failed_logins_list = []
brute_force_counts = {}
high_critical_events = []
suspicious_foreign_logins = []

# ==========================================
# PHASE 3: PARSING THE EVIDENCE
# ==========================================
# PYTHON: Open the CSV file in read-only mode ('r').
# SOC: We read the logs to extract the Indicators of Compromise (IoCs).
with open(input_file, mode='r') as csvfile:
    reader = csv.DictReader(csvfile)
    
    # PYTHON: Loop through every single log entry.
    for row in reader:
        # PYTHON: Increment total events.
        total_events += 1
        
        # PYTHON: Categorize the action.
        action = row['action'].lower()
        if action == 'success':
            success_events += 1
        elif action == 'failed':
            failed_events += 1
        elif action == 'blocked' or action == 'detected':
            blocked_events += 1
            
        # PYTHON: Task 3 - Failed Login Detection
        event_type = row['event_type'].lower()
        if event_type == 'login' and action == 'failed':
            failed_logins_list.append({
                'timestamp': row['timestamp'],
                'source_ip': row['source_ip'],
                'username': row['username'],
                'country': row['country']
            })
            
            # PYTHON: Task 4 - Brute Force Detection (Counting IPs)
            ip = row['source_ip']
            brute_force_counts[ip] = brute_force_counts.get(ip, 0) + 1
            
        # PYTHON: Task 5 - Triage High/Critical Alerts
        severity = row['severity'].lower()
        if severity in ['high', 'critical']:
            high_critical_events.append({
                'timestamp': row['timestamp'],
                'username': row['username'],
                'event_type': row['event_type'],
                'severity': row['severity']
            })
            
        # PYTHON: Task 6 - Suspicious Geolocation (Non-US)
        if event_type == 'login' and row['country'] != 'US':
            suspicious_foreign_logins.append({
                'source_ip': row['source_ip'],
                'username': row['username'],
                'country': row['country'],
                'action': row['action']
            })

# ==========================================
# PHASE 4: EXPORTING THE REPORT TO A TXT FILE
# ==========================================
# PYTHON: Open our output file in 'w' (write) mode. 
# SOC: This creates a new file. If the file already exists, 'w' mode will overwrite it with fresh data.
with open(output_file, mode='w') as report:
    
    # PYTHON: Use .write() instead of print(). We MUST add '\n' to force a new line.
    # SOC: Creating a clear header for the first section of our evidence.
    report.write("--- Task 3: Failed Logins ---\n")
    
    # PYTHON: Loop through our failed logins.
    for entry in failed_logins_list:
        # PYTHON: Write the formatted string to the file, ending with \n.
        report.write(f"Timestamp: {entry['timestamp']}, IP: {entry['source_ip']}, User: {entry['username']}, Country: {entry['country']}\n")

    # PYTHON: Write the Brute Force header.
    report.write("\n--- Task 4: Brute Force Detection ---\n")
    
    # PYTHON: Create the list for IPs that crossed the threshold.
    brute_force_ips = []
    
    # PYTHON: Evaluate the dictionary of failed attempts.
    for ip, count in brute_force_counts.items():
        # SOC: If attempts are 3 or more, flag it as an attack.
        if count >= 3:
            report.write(f"Brute Force Attempt: {ip} - Attempts: {count}\n")
            brute_force_ips.append(ip)

    # PYTHON: Write the High/Critical header.
    report.write("\n--- Task 5: High/Critical Severity ---\n")
    
    # PYTHON: Export the severe alerts for immediate Incident Response review.
    for entry in high_critical_events:
        report.write(f"Timestamp: {entry['timestamp']}, User: {entry['username']}, Event: {entry['event_type']}, Severity: {entry['severity']}\n")

    # PYTHON: Write the Suspicious Logins header.
    report.write("\n--- Task 6: Suspicious Foreign Logins ---\n")
    
    # PYTHON: Export the geo-fencing violations.
    for entry in suspicious_foreign_logins:
        report.write(f"IP: {entry['source_ip']}, User: {entry['username']}, Country: {entry['country']}, Action: {entry['action']}\n")

    # PYTHON: Write the Executive Summary at the very end.
    # SOC: This is the TL;DR (Too Long; Didn't Read) for the Chief Information Security Officer (CISO).
    report.write("\n========== SECURITY REPORT ==========\n")
    report.write(f"Total Events: {total_events}\n")
    report.write(f"Failed Logins: {len(failed_logins_list)}\n")
    report.write(f"Brute Force IPs: {len(brute_force_ips)}\n")
    report.write(f"High/Critical Events: {len(high_critical_events)}\n")
    report.write(f"Suspicious Foreign Logins: {len(suspicious_foreign_logins)}\n")
    report.write("=====================================\n")

# PYTHON: Print a final confirmation to the console so the user knows the script finished successfully.
print(f"Analysis complete! Evidence successfully exported to {output_file}")

Analysis complete! Evidence successfully exported to SOC_Investigation_Report.txt


In [3]:
# PYTHON: Bring in our CSV reading tool.
# SOC: We need our SIEM translator goggles to read the structured firewall logs.
import csv

# PYTHON: Store the location of our evidence file.
# SOC: This is the raw network traffic we are investigating.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP THE DETECTIVE's MEMORY
# ==========================================

# PYTHON: Create an empty dictionary {} to hold numbers.
# SOC: This is our whiteboard. We write down an IP address and put tally marks next to it every time it fails a login.
ip_fail_tracker = {}

# PYTHON: Create an empty 'set'. A set is like a list, but it only holds unique items (no duplicates).
# SOC: This is our "Blacklist." If an IP gets 3 tally marks on the whiteboard, its name gets moved to this permanent Blacklist.
brute_force_blacklist = set()

# PYTHON: Create an empty list [] for our absolute worst-case scenarios.
# SOC: This is the "Code Red" folder. If an IP on the Blacklist gets a successful login, the evidence goes in here.
confirmed_compromises = []

# ==========================================
# PHASE 2: CHRONOLOGICAL INVESTIGATION
# ==========================================

# PYTHON: Open the file securely in read-only mode ('r').
# SOC: We are opening the timeline. Logs are written chronologically, so we are traveling forward through time as we read top-to-bottom.
with open(file_path, mode='r') as file:
    
    # PYTHON: Use DictReader to turn every row into a labeled package.
    # SOC: Standardizing the evidence so we can pull exactly what we want.
    reader = csv.DictReader(file)
    
    # PYTHON: Start looping through every single row, one by one.
    # SOC: The detective slides their finger down the page, reading event by event.
    for row in reader:
        
        # PYTHON: Extract the exact details we need into clean, lowercase variables.
        # SOC: We are pulling the "What" (event), "Result" (action), "Who" (user), and "Where" (IP).
        event_type = row['event_type'].lower()
        action = row['action'].lower()
        source_ip = row['source_ip']
        username = row['username']
        timestamp = row['timestamp']
        
        # PYTHON: Check if this specific row is a login attempt.
        # SOC: We ignore file downloads or port scans for this script. We only care about front-door logins.
        if event_type == 'login':
            
            # --------------------------------------
            # THE TRAP: COUNTING FAILURES
            # --------------------------------------
            # PYTHON: Check if the login was a failure.
            # SOC: Did they enter the wrong password?
            if action == 'failed':
                
                # PYTHON: Look at the whiteboard. Add 1 tally mark to this IP. If it's a new IP, start it at 1.
                # SOC: Documenting the failed attempt.
                ip_fail_tracker[source_ip] = ip_fail_tracker.get(source_ip, 0) + 1
                
                # PYTHON: Check if the tally marks for this IP have reached 3 or more.
                # SOC: 1 fail is a typo. 3 fails is an attacker. Have they crossed the threshold?
                if ip_fail_tracker[source_ip] >= 3:
                    
                    # PYTHON: Add this IP to our Set.
                    # SOC: The threshold is crossed! We permanently write this IP address onto the "Known Attacker Blacklist".
                    brute_force_blacklist.add(source_ip)
            
            # --------------------------------------
            # THE TRIGGER: DETECTING THE BREACH
            # --------------------------------------
            # PYTHON: Alternatively, check if the login was a SUCCESS.
            # SOC: The firewall just let someone in. Now we must ask a critical question...
            elif action == 'success':
                
                # PYTHON: Is this 'source_ip' currently sitting on our 'brute_force_blacklist'?
                # SOC: "Wait a minute! Did the firewall just let in one of the known attackers from our blacklist?!"
                if source_ip in brute_force_blacklist:
                    
                    # PYTHON: If YES, copy the row details into our Code Red list.
                    # SOC: ALARM TRIPPED! The attacker guessed the password. The network is compromised. Save the evidence!
                    confirmed_compromises.append({
                        'timestamp': timestamp,
                        'attacker_ip': source_ip,
                        'compromised_account': username
                    })

# ==========================================
# PHASE 3: THE INCIDENT RESPONSE REPORT
# ==========================================

# PYTHON: Print a massive warning header to the terminal.
# SOC: This demands immediate attention from the Incident Response team.
print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
print("!!! CODE RED: CONFIRMED ACCOUNT COMPROMISE !!!")
print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

# PYTHON: Check if our Code Red folder is empty (Length is 0).
# SOC: Did we catch anyone?
if len(confirmed_compromises) == 0:
    # PYTHON: Print the all-clear.
    # SOC: The network is safe. No brute-force attackers successfully broke in today.
    print("STATUS: SECURE. No successful brute-force breaches detected.")

# PYTHON: If the folder is NOT empty...
else:
    # PYTHON: Print exactly how many breaches we found.
    # SOC: Telling management the scale of the disaster.
    print(f"CRITICAL ALERT: {len(confirmed_compromises)} accounts were successfully breached after a brute-force attack!\n")
    
    # PYTHON: Loop through the red alerts and print the details.
    # SOC: Handing the exact time, attacker IP, and victim username to the Incident Responders so they can instantly lock the accounts.
    for breach in confirmed_compromises:
        print(f"-> [TIME]: {breach['timestamp']}")
        print(f"-> [ATTACKER IP]: {breach['attacker_ip']} breached the network.")
        print(f"-> [VICTIM ACCOUNT]: The account '{breach['compromised_account']}' must be locked IMMEDIATELY.\n")


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!! CODE RED: CONFIRMED ACCOUNT COMPROMISE !!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

CRITICAL ALERT: 1 accounts were successfully breached after a brute-force attack!

-> [TIME]: 2024-02-03 11:10:12
-> [ATTACKER IP]: 203.0.113.45 breached the network.
-> [VICTIM ACCOUNT]: The account 'jsmith' must be locked IMMEDIATELY.



In [4]:
# PYTHON: Import our trusty CSV reader tool.
# SOC: We need this to parse our structured firewall evidence.
import csv

# PYTHON: Store the location of our evidence file.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING THE TRIPWIRES
# ==========================================

# PYTHON: Define a constant integer for our byte threshold (5,000,000 bytes = ~5 Megabytes).
# SOC: In a real enterprise, we might set this to 500MB. For our specific logs, we are looking for anomalies over 5MB. This is our "Heavy Load" tripwire.
EXFILTRATION_THRESHOLD = 5000000 

# PYTHON: Create an empty list to store the evidence of any theft.
# SOC: This is our "Stolen Goods" folder. If someone trips the wire, their log goes here.
exfiltration_alerts = []

# ==========================================
# PHASE 2: SCANNING THE PERIMETER
# ==========================================

# PYTHON: Open the file securely in read-only mode ('r').
# SOC: Opening the network traffic logs for inspection.
with open(file_path, mode='r') as file:
    
    # PYTHON: Use DictReader to turn every row into a dictionary.
    reader = csv.DictReader(file)
    
    # PYTHON: Start looping through every single log.
    for row in reader:
        
        # PYTHON: Extract the details we need. 
        # SOC: We want to know "Who" (username), "What" (event_type), "How much" (bytes_out), and the context (message).
        event_type = row['event_type'].lower()
        action = row['action'].lower()
        username = row['username']
        timestamp = row['timestamp']
        message = row['message']
        
        # PYTHON: Extract 'bytes_out' safely. If the column is completely blank, default it to '0'.
        # SOC: Network logs can be messy. Sometimes a blocked connection has no byte data at all. We must handle blanks so our script doesn't crash.
        raw_bytes_out = row.get('bytes_out', '0')
        if raw_bytes_out == '':
            raw_bytes_out = '0'
            
        # PYTHON: Convert the text string of bytes into an actual mathematical integer.
        # SOC: The CSV reader sees "5242880" as a word. We must turn it into a number so we can use the "greater than" (>) math symbol later.
        bytes_out = int(raw_bytes_out)
        
        # --------------------------------------
        # THE TRAP: CATCHING THE THEFT
        # --------------------------------------
        
        # PYTHON: Rule 1 - Did the action actually succeed?
        # SOC: If the firewall BLOCKED the 5MB download, we are safe. We only panic if the action was 'success'.
        if action == 'success':
            
            # PYTHON: Rule 2 - Is this a file access or a database query?
            # SOC: We expect big downloads for things like "video_stream". We DO NOT expect massive downloads for simple database queries.
            if event_type == 'database_query' or event_type == 'file_access':
                
                # PYTHON: Rule 3 - Did the data leaving the network cross our tripwire?
                # SOC: "Did this person just walk out the door carrying more than 5 Megabytes of database files?"
                if bytes_out > EXFILTRATION_THRESHOLD:
                    
                    # PYTHON: All rules met! Add the evidence to our alert list.
                    # SOC: RED ALERT. Data is actively being stolen. Save the context immediately.
                    exfiltration_alerts.append({
                        'timestamp': timestamp,
                        'user': username,
                        'event': event_type,
                        'bytes_stolen': bytes_out,
                        'details': message
                    })

# ==========================================
# PHASE 3: THE DATA LOSS PREVENTION (DLP) REPORT
# ==========================================

# PYTHON: Print a severe warning header to the terminal.
print("\n***************************************************")
print("*** SEV-1 ALERT: MASSIVE DATA EXFILTRATION DETECTED ***")
print("***************************************************\n")

# PYTHON: Check if our Stolen Goods folder is empty.
if len(exfiltration_alerts) == 0:
    # PYTHON: Print the all-clear.
    # SOC: No massive database dumps occurred. The company's data is safe.
    print("STATUS: SECURE. No large-scale data theft detected.")

# PYTHON: If the folder is NOT empty...
else:
    # PYTHON: Print the total number of theft events.
    print(f"CRITICAL: {len(exfiltration_alerts)} massive data transfers detected!\n")
    
    # PYTHON: Loop through the alerts and print exactly what was taken.
    # SOC: The Incident Response team needs to know exactly WHICH database was stolen so they can notify the right customers.
    for alert in exfiltration_alerts:
        # PYTHON: Do a quick math calculation to turn bytes into Megabytes (MB) for human readability.
        mb_stolen = round(alert['bytes_stolen'] / 1000000, 2)
        
        print(f"-> [TIME]: {alert['timestamp']}")
        print(f"-> [SUSPECT]: User '{alert['user']}'")
        print(f"-> [ACTION]: {alert['event']}")
        print(f"-> [DATA LOST]: {mb_stolen} MB transferred out of the network!")
        print(f"-> [CONTEXT]: {alert['details']}\n")


***************************************************
*** SEV-1 ALERT: MASSIVE DATA EXFILTRATION DETECTED ***
***************************************************

CRITICAL: 1 massive data transfers detected!

-> [TIME]: 2024-02-03 11:50:00
-> [SUSPECT]: User 'jsmith'
-> [ACTION]: database_query
-> [DATA LOST]: 5.24 MB transferred out of the network!
-> [CONTEXT]: CRITICAL: 5MB database dump - all tables exported



In [5]:
# PYTHON: Import our CSV reading tool.
# SOC: We need this to read the structured network traffic logs.
import csv

# PYTHON: Store the evidence file location.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP THE RADAR
# ==========================================

# PYTHON: Define our threshold integer. 
# SOC: Normal users usually only talk to 1 or 2 servers a day (like the email server and the file share). If one computer suddenly talks to 5 DIFFERENT servers, they are scanning the network.
LATERAL_THRESHOLD = 5 

# PYTHON: Create an empty dictionary. 
# SOC: This whiteboard will map ONE Source IP to MANY Destination IPs.
# Example: '192.168.1.105' -> {'10.0.0.50', '10.0.0.51', '10.0.0.52'}
lateral_tracker = {}

# PYTHON: Create a list to hold the final alerts.
lateral_alerts = []

# ==========================================
# PHASE 2: CHARTING THE CONNECTIONS
# ==========================================

# PYTHON: Open the file securely in read-only mode ('r').
# SOC: Analyzing the traffic flow of the entire network.
with open(file_path, mode='r') as file:
    
    # PYTHON: Convert the CSV rows into manageable dictionaries.
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through every log entry chronologically.
    for row in reader:
        
        # PYTHON: Extract the details. We need the Source, the Destination, and the Action.
        source_ip = row['source_ip']
        destination_ip = row['destination_ip']
        action = row['action'].lower()
        username = row['username']
        
        # --------------------------------------
        # THE TRAP: MAPPING THE NETWORK
        # --------------------------------------
        
        # PYTHON: Rule 1 - We only care if the connection was actually 'success'ful or 'allowed'.
        # SOC: If the firewall blocked them, they didn't successfully move laterally. We track the successful jumps.
        if action == 'success' or action == 'allowed':
            
            # PYTHON: Initialize the dictionary key if it doesn't exist yet.
            # SOC: If this is the very first time we have seen this Source IP, create a blank "Set" for it. A "Set" is a special Python list that automatically deletes duplicates.
            if source_ip not in lateral_tracker:
                lateral_tracker[source_ip] = set()
                
            # PYTHON: Add the Destination IP to the Source IP's set.
            # SOC: We draw a line on our map from the Source to the Destination. If they connect to the same server 100 times, the "Set" keeps it as just 1 line (unique connections only).
            lateral_tracker[source_ip].add(destination_ip)

# ==========================================
# PHASE 3: EVALUATING THE EVIDENCE
# ==========================================

# PYTHON: The loop is finished reading the file. Now we analyze our completed map.
# Loop through our tracker, unpacking the Source IP and the Set of Destinations it talked to.
for ip, destinations in lateral_tracker.items():
    
    # PYTHON: Check the length (len) of the set. How many UNIQUE computers did this IP talk to?
    # SOC: Did they cross our threshold of 5 different servers?
    if len(destinations) >= LATERAL_THRESHOLD:
        
        # PYTHON: If YES, add them to our alert list.
        # SOC: This computer is infected and crawling through the network. We must isolate it.
        lateral_alerts.append({
            'infected_ip': ip,
            'unique_targets': len(destinations),
            'target_list': destinations
        })

# ==========================================
# PHASE 4: THE INCIDENT RESPONSE REPORT
# ==========================================

# PYTHON: Print the warning header.
print("\n=======================================================")
print("=== THREAT HUNT: LATERAL MOVEMENT DETECTED ===")
print("=======================================================\n")

# PYTHON: If our alert list is empty, print the all-clear.
if len(lateral_alerts) == 0:
    print("STATUS: SECURE. No internal network crawling detected.")

# PYTHON: If we found attackers...
else:
    print(f"CRITICAL: {len(lateral_alerts)} machines are exhibiting worm-like lateral movement!\n")
    
    # PYTHON: Loop through the alerts and expose the behavior.
    for alert in lateral_alerts:
        print(f"-> [COMPROMISED MACHINE]: {alert['infected_ip']}")
        print(f"-> [BEHAVIOR]: Successfully connected to {alert['unique_targets']} different internal servers.")
        
        # PYTHON: Join the set of IPs into a readable string so the IR team knows exactly what was compromised.
        # SOC: The IR team will now have to go check every single one of these destination IPs to see if the hacker planted malware on them.
        targets_formatted = ", ".join(alert['target_list'])
        print(f"-> [TARGETS HIT]: {targets_formatted}\n")


=== THREAT HUNT: LATERAL MOVEMENT DETECTED ===

STATUS: SECURE. No internal network crawling detected.


In [6]:
# PYTHON: Import our CSV reader.
import csv
# PYTHON: Import datetime. This translates text like '2024-02-03 08:15:23' into actual "Time" that Python can do math with.
# SOC: We need to calculate the exact time difference between two logins.
from datetime import datetime

file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP THE PASSPORT TRACKER
# ==========================================

# PYTHON: Create an empty dictionary to store the last known location of every user.
# SOC: This is our Passport Control book. 
# Format will be: {'jsmith': {'time': [Time Object], 'country': 'US', 'ip': '192.168.1.105'}}
user_last_known_location = {}

# PYTHON: Create an empty list to store the caught hackers.
impossible_travel_alerts = []

# ==========================================
# PHASE 2: AUDITING THE FLIGHT LOGS
# ==========================================

with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    for row in reader:
        event_type = row['event_type'].lower()
        action = row['action'].lower()
        username = row['username']
        current_country = row['country']
        current_ip = row['source_ip']
        raw_timestamp = row['timestamp']
        
        # PYTHON: Rule 1 - We only care about SUCCESSFUL logins. 
        # SOC: A failed login from China doesn't mean the account is compromised, it just means they are guessing. A successful login means they are in.
        if event_type == 'login' and action == 'success':
            
            # PYTHON: Convert the raw text timestamp into a Python 'datetime' object.
            # The '%Y-%m-%d %H:%M:%S' tells Python exactly how our text is formatted (Year-Month-Day Hour:Minute:Second).
            try:
                current_time = datetime.strptime(raw_timestamp, '%Y-%m-%d %H:%M:%S')
            except ValueError:
                # If the log is corrupted and the time is formatted wrong, skip this row.
                continue
                
            # --------------------------------------
            # THE TRAP: CHECKING THE PASSPORT
            # --------------------------------------
            
            # PYTHON: Check if we have seen this user log in before.
            if username in user_last_known_location:
                
                # PYTHON: Retrieve their last known login details.
                last_login = user_last_known_location[username]
                
                # PYTHON: Did their country change? (And ensure neither country is blank).
                if current_country != last_login['country'] and current_country != '' and last_login['country'] != '':
                    
                    # PYTHON: Math time! Subtract the old time from the new time.
                    # SOC: How long has it been since they last logged in?
                    time_diff = current_time - last_login['time']
                    
                    # PYTHON: Convert the difference into Hours.
                    hours_diff = time_diff.total_seconds() / 3600
                    
                    # PYTHON: The "Impossible" Rule. If they changed countries in less than 12 hours, trigger the alarm.
                    # SOC: Commercial flights take time. You cannot get from the US to Asia or Europe in 2 hours.
                    if hours_diff < 12:
                        impossible_travel_alerts.append({
                            'username': username,
                            'old_country': last_login['country'],
                            'old_time': last_login['time'].strftime('%Y-%m-%d %H:%M:%S'),
                            'new_country': current_country,
                            'new_time': raw_timestamp,
                            'hours_between': round(hours_diff, 2), # Round to 2 decimal places
                            'hacker_ip': current_ip
                        })
            
            # PYTHON: Update the passport book! Overwrite the old location with the newest location.
            # SOC: We always want to compare the NEXT login to this current one.
            user_last_known_location[username] = {
                'time': current_time,
                'country': current_country,
                'ip': current_ip
            }

# ==========================================
# PHASE 3: THE GLOBAL THREAT REPORT
# ==========================================

print("\n🌍====================================================🌍")
print("🌍=== CRITICAL: IMPOSSIBLE TRAVEL ALERTS DETECTED  ===🌍")
print("🌍====================================================🌍\n")

if len(impossible_travel_alerts) == 0:
    print("STATUS: SECURE. All users are logging in from logical physical locations.")
else:
    print(f"URGENT: {len(impossible_travel_alerts)} compromised accounts detected due to impossible travel!\n")
    
    for alert in impossible_travel_alerts:
        print(f"-> [COMPROMISED USER]: {alert['username']}")
        print(f"     Location 1: {alert['old_country']} at {alert['old_time']}")
        print(f"     Location 2: {alert['new_country']} at {alert['new_time']}")
        print(f"-> [ANOMALY]: The user traveled between {alert['old_country']} and {alert['new_country']} in only {alert['hours_between']} hours!")
        print(f"-> [ACTION]: Lock account '{alert['username']}' and block IP {alert['hacker_ip']} immediately.\n")


🌍====================================================🌍
🌍=== CRITICAL: IMPOSSIBLE TRAVEL ALERTS DETECTED  ===🌍
🌍====================================================🌍

URGENT: 1 compromised accounts detected due to impossible travel!

-> [COMPROMISED USER]: jsmith
     Location 1: US at 2024-02-03 09:50:00
     Location 2: CN at 2024-02-03 11:10:12
-> [ANOMALY]: The user traveled between US and CN in only 1.34 hours!
-> [ACTION]: Lock account 'jsmith' and block IP 203.0.113.45 immediately.



In [7]:
# PYTHON: Import our CSV reader tool.
import csv
# PYTHON: Import the datetime tool so Python can understand "Time" and "Clocks".
from datetime import datetime

# PYTHON: Set the evidence file path.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING THE BUSINESS CLOCK
# ==========================================

# PYTHON: Define our standard business hours using 24-hour (military) time.
# SOC: We define when the "doors" to the network should be open. 
# 6 means 6:00 AM. 20 means 8:00 PM. 
BUSINESS_START_HOUR = 6
BUSINESS_END_HOUR = 20

# PYTHON: Create an empty list to store the night-time intruders.
# SOC: This is the folder for our "After Hours" alerts.
off_hours_alerts = []

# ==========================================
# PHASE 2: REVIEWING THE SECURITY TAPES
# ==========================================

with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    for row in reader:
        # PYTHON: Extract the details of the event.
        event_type = row['event_type'].lower()
        action = row['action'].lower()
        username = row['username']
        raw_timestamp = row['timestamp']
        source_ip = row['source_ip']
        
        # --------------------------------------
        # THE TRAP: FILTERING FOR SUCCESS
        # --------------------------------------
        
        # PYTHON: Rule 1 - We only care if the action was 'success' or 'allowed'.
        # SOC: If a hacker tries to log in at 3 AM and fails, the firewall blocked them. But if they SUCCEED at 3 AM, we have a major incident.
        if action == 'success' or action == 'allowed':
            
            # PYTHON: Safely convert the text timestamp into a real Time object.
            try:
                event_time = datetime.strptime(raw_timestamp, '%Y-%m-%d %H:%M:%S')
            except ValueError:
                # Skip broken log lines
                continue
            
            # PYTHON: Extract JUST the hour from the full timestamp.
            # SOC: If the time is '03:15:22', event_hour becomes the number 3.
            event_hour = event_time.hour
            
            # --------------------------------------
            # THE TRIGGER: CHECKING THE CLOCK
            # --------------------------------------
            
            # PYTHON: Rule 2 - The "Or" Statement. Is the hour LESS THAN 6 (Midnight to 5:59 AM) OR GREATER THAN/EQUAL TO 20 (8:00 PM to 11:59 PM)?
            # SOC: Is the office closed right now?
            if event_hour < BUSINESS_START_HOUR or event_hour >= BUSINESS_END_HOUR:
                
                # PYTHON: Rule broken! Add the evidence to our alert list.
                off_hours_alerts.append({
                    'timestamp': raw_timestamp,
                    'username': username,
                    'event': event_type,
                    'ip': source_ip,
                    'hour': event_hour
                })

# ==========================================
# PHASE 3: THE NIGHT WATCHMAN REPORT
# ==========================================

print("\n🌙====================================================🌙")
print("🌙=== ALERT: OFF-HOURS NETWORK ACTIVITY DETECTED   ===🌙")
print("🌙====================================================🌙\n")

# PYTHON: Check if our after-hours folder is empty.
if len(off_hours_alerts) == 0:
    print("STATUS: SECURE. No successful actions occurred outside of business hours.")
else:
    print(f"WARNING: {len(off_hours_alerts)} successful events happened while the office was closed!\n")
    
    # PYTHON: Loop through the alerts and expose the behavior.
    for alert in off_hours_alerts:
        
        # PYTHON: A quick formatting trick to make 24-hour time look like 12-hour AM/PM time for the report.
        display_hour = alert['hour']
        am_pm = "AM" if display_hour < 12 else "PM"
        if display_hour > 12:
            display_hour -= 12
        elif display_hour == 0:
            display_hour = 12
            
        print(f"-> [TIME OF EVENT]: {alert['timestamp']} (Occurred during the {display_hour} {am_pm} hour)")
        print(f"-> [USER]: '{alert['username']}' via IP {alert['ip']}")
        print(f"-> [ACTION TAKEN]: Successfully executed a '{alert['event']}'")
        print("-> [RECOMMENDATION]: Verify with the employee if they were working late/early. If not, isolate the machine.\n")


🌙====================================================🌙
🌙=== ALERT: OFF-HOURS NETWORK ACTIVITY DETECTED   ===🌙
🌙====================================================🌙

STATUS: SECURE. No successful actions occurred outside of business hours.


In [8]:
# PYTHON: Import the CSV module to read our log file.
import csv

# PYTHON: Store the evidence file path.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP THE SENSORS
# ==========================================

# PYTHON: Define our threshold integer.
# SOC: A normal computer might talk to 2 or 3 ports (like Web and Email). If a single computer tries to talk to 10 DIFFERENT ports, it is a scanner. 
SCAN_THRESHOLD = 10

# PYTHON: Create an empty dictionary to track the behavior.
# SOC: We will use this whiteboard to map a Source IP to a 'Set' of the ports they knock on.
# Example: '185.220.101.33' -> {'22', '80', '443', '3389', '8080'...}
scanner_tracker = {}

# PYTHON: Create an empty list for the final alerts.
# SOC: This folder holds the confirmed attackers we need to block.
port_scan_alerts = []

# ==========================================
# PHASE 2: MONITORING THE DOORS (PORTS)
# ==========================================

# PYTHON: Open the file securely in read-only mode ('r').
with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through every log entry.
    for row in reader:
        
        # PYTHON: Extract the details we need. 
        # SOC: We need the attacker's IP and the specific door (port) they knocked on.
        source_ip = row['source_ip']
        port = row['port']
        
        # --------------------------------------
        # THE TRAP: COUNTING UNIQUE KNOCKS
        # --------------------------------------
        
        # PYTHON: Rule 1 - Ensure the port column isn't totally blank.
        # SOC: Sometimes firewalls log weird internal errors that don't have a port. We skip those.
        if port != '':
            
            # PYTHON: If this IP is not on our whiteboard yet, add it and give it an empty 'Set'.
            # SOC: A 'Set' automatically ignores duplicates. If they knock on Port 80 five times, the Set only counts it as '1' unique door.
            if source_ip not in scanner_tracker:
                scanner_tracker[source_ip] = set()
                
            # PYTHON: Add the port to the IP's set.
            scanner_tracker[source_ip].add(port)

# ==========================================
# PHASE 3: EVALUATING THE EVIDENCE
# ==========================================

# PYTHON: The file reading is done. Now we review our whiteboard.
# Loop through the dictionary, unpacking the IP and the set of ports.
for ip, ports_knocked in scanner_tracker.items():
    
    # PYTHON: Rule 2 - Did the length of the set cross our threshold?
    # SOC: "Did this guy just jiggle 10 different door handles?"
    if len(ports_knocked) >= SCAN_THRESHOLD:
        
        # PYTHON: Threshold crossed! Add them to the alert list.
        # SOC: We caught a scout. Save their IP so we can block them before they launch a real attack.
        port_scan_alerts.append({
            'attacker_ip': ip,
            'total_doors_checked': len(ports_knocked),
            # Convert the set to a sorted list of numbers so it's easy for humans to read
            'doors_list': sorted([int(p) for p in ports_knocked]) 
        })

# ==========================================
# PHASE 4: THE RECONNAISSANCE REPORT
# ==========================================

# PYTHON: Print the warning header.
print("\n🔎======================================================🔎")
print("🔎=== EARLY WARNING: PORT SCANNING BEHAVIOR DETECTED ===🔎")
print("🔎======================================================🔎\n")

# PYTHON: If the alert folder is empty, we are safe.
if len(port_scan_alerts) == 0:
    print("STATUS: SECURE. No aggressive network scanning detected.")

# PYTHON: If we caught scanners...
else:
    print(f"WARNING: {len(port_scan_alerts)} rogue IP addresses are probing our network defenses!\n")
    
    # PYTHON: Loop through the alerts and print exactly what happened.
    for alert in port_scan_alerts:
        print(f"-> [HOSTILE SCOUT]: IP Address {alert['attacker_ip']}")
        print(f"-> [BEHAVIOR]: Attempted to connect to {alert['total_doors_checked']} different ports.")
        
        # PYTHON: Turn the list of ports into a nice, comma-separated string.
        # SOC: The IR team will look at this list to figure out what the hacker was looking for. Were they looking for web servers? Email servers? Databases?
        ports_formatted = ", ".join(map(str, alert['doors_list']))
        print(f"-> [PORTS PROBED]: {ports_formatted}")
        print("-> [ACTION]: Add this IP to the Firewall Blocklist immediately.\n")


🔎======================================================🔎
🔎=== EARLY WARNING: PORT SCANNING BEHAVIOR DETECTED ===🔎
🔎======================================================🔎

STATUS: SECURE. No aggressive network scanning detected.


In [9]:
# PYTHON: Import the CSV module to read our security logs.
import csv

# PYTHON: Store the evidence file path.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: DEFINING THE THREAT INTEL
# ==========================================

# PYTHON: Create a 'Set' of known malicious ports.
# SOC: Threat Intelligence tells us hackers love specific ports. 
# 4444 is the default for Metasploit (a famous hacking tool). 
# 1337 and 31337 are old-school "leet" hacker ports. 
# 6667 is often used for malicious IRC botnets.
KNOWN_BAD_PORTS = {'4444', '1337', '31337', '6667', '4445'}

# PYTHON: Create an empty list to hold our critical alerts.
# SOC: If we find a reverse shell, it is an immediate Sev-1 Incident.
backdoor_alerts = []

# ==========================================
# PHASE 2: HUNTING FOR THE BACKDOORS
# ==========================================

# PYTHON: Open the file securely in read-only mode.
with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through every log entry chronologically.
    for row in reader:
        
        # PYTHON: Extract the details of the network connection.
        timestamp = row['timestamp']
        source_ip = row['source_ip']
        username = row['username']
        event_type = row['event_type'].lower()
        port = row['port']
        action = row['action'].lower()
        message = row['message']
        
        # --------------------------------------
        # THE TRAP: CHECKING THE DOORS
        # --------------------------------------
        
        # PYTHON: Rule 1 - Is the traffic using one of our known hacker ports?
        # SOC: If any traffic at all touches port 4444, we need to know about it immediately.
        used_bad_port = port in KNOWN_BAD_PORTS
        
        # PYTHON: Rule 2 - Did the firewall actually classify this as a reverse shell?
        # SOC: Sometimes the firewall is smart enough to detect the payload, regardless of what port it uses.
        is_reverse_shell = (event_type == 'reverse_shell')
        
        # PYTHON: The "OR" Trigger. If Rule 1 OR Rule 2 is True...
        if used_bad_port or is_reverse_shell:
            
            # PYTHON: Add the evidence to our critical alert folder.
            # SOC: We also want to capture the 'action' (was it blocked or allowed?) and the raw 'message' for context.
            backdoor_alerts.append({
                'timestamp': timestamp,
                'infected_host': source_ip,
                'user': username,
                'port': port,
                'status': action,
                'context': message
            })

# ==========================================
# PHASE 3: THE INCIDENT RESPONSE REPORT
# ==========================================

# PYTHON: Print a massive red-alert header.
print("\n☠️=============================================================☠️")
print("☠️=== CRITICAL: REVERSE SHELL / MALICIOUS PORT DETECTED ===☠️")
print("☠️=============================================================☠️\n")

# PYTHON: Check if our alert folder is empty.
if len(backdoor_alerts) == 0:
    print("STATUS: SECURE. No malicious backdoors or non-standard ports found.")

# PYTHON: If we found backdoors...
else:
    print(f"URGENT: {len(backdoor_alerts)} backdoor connection attempts detected!\n")
    
    # PYTHON: Loop through the alerts and print exactly what happened.
    for alert in backdoor_alerts:
        
        # PYTHON: Check if the connection was successful to determine the severity of the message.
        if alert['status'] == 'success' or alert['status'] == 'allowed':
            print(f"-> [BREACH CONFIRMED]: The backdoor is ACTIVE on host {alert['infected_host']}!")
        else:
            print(f"-> [ATTACK BLOCKED]: A backdoor attempted to open on {alert['infected_host']}, but was blocked.")
            
        print(f"   Time: {alert['timestamp']}")
        print(f"   User: {alert['user']}")
        print(f"   Port Used: {alert['port']}")
        print(f"   Firewall Log: '{alert['context']}'")
        print("   Action Required: Disconnect this machine from the network immediately.\n")


☠️=============================================================☠️
☠️=== CRITICAL: REVERSE SHELL / MALICIOUS PORT DETECTED ===☠️
☠️=============================================================☠️

URGENT: 1 backdoor connection attempts detected!

-> [ATTACK BLOCKED]: A backdoor attempted to open on 203.0.113.45, but was blocked.
   Time: 2024-02-03 11:40:45
   User: jsmith
   Port Used: 4444
   Firewall Log: 'Reverse shell connection detected on port 4444'
   Action Required: Disconnect this machine from the network immediately.



In [10]:
# PYTHON: Import the CSV module to read our security logs.
import csv

# PYTHON: Store the evidence file path.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: DEFINING THE BOT LIST
# ==========================================

# PYTHON: Create a list of known automated tools and hacking scripts.
# SOC: Threat Intel tells us these are standard command-line tools. 
# 'curl' and 'wget' are used to download payloads. 
# 'python-requests' means a Python script is attacking us.
# 'nmap' and 'sqlmap' are literal hacking tools.
SUSPICIOUS_TOOLS = ['curl', 'wget', 'python-requests', 'nmap', 'sqlmap', 'nikto', 'powershell']

# PYTHON: Create an empty list to store our bot alerts.
# SOC: This folder will hold all the traffic generated by non-human robots.
bot_alerts = []

# ==========================================
# PHASE 2: CHECKING THE ID BADGES
# ==========================================

# PYTHON: Open the file securely in read-only mode.
with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through every log entry.
    for row in reader:
        
        # PYTHON: Extract the details.
        timestamp = row['timestamp']
        source_ip = row['source_ip']
        port = row['port']
        action = row['action'].lower()
        
        # PYTHON: Extract the user agent. Use .get() to safely handle cases where the column might be missing, and make it lowercase.
        user_agent = row.get('user_agent', '').lower()
        
        # --------------------------------------
        # THE TRAP: ANALYZING THE BADGE
        # --------------------------------------
        
        # PYTHON: Create a "Flag" (a True/False variable) and start it as False.
        # SOC: Innocent until proven guilty.
        is_bot = False
        detected_tool = "Unknown"
        
        # PYTHON: Loop through our list of known bad tools to see if any of their names are hiding inside the User-Agent string.
        for tool in SUSPICIOUS_TOOLS:
            if tool in user_agent:
                is_bot = True
                detected_tool = tool
                break # PYTHON: We found a match, no need to keep checking the rest of the list.
                
        # PYTHON: Rule 2 - The "Blank Badge" Check.
        # SOC: If someone connects to a Web Port (80 for HTTP or 443 for HTTPS) but has NO User-Agent, it is almost always a malicious script. Normal web browsers NEVER leave this blank.
        if user_agent == '' and (port == '80' or port == '443'):
            is_bot = True
            detected_tool = "Blank/Missing User-Agent"
            
        # --------------------------------------
        # THE TRIGGER: CATCHING THE ROBOTS
        # --------------------------------------
        
        # PYTHON: If our 'is_bot' flag got flipped to True...
        if is_bot == True:
            
            # PYTHON: Add the evidence to our alert folder.
            bot_alerts.append({
                'timestamp': timestamp,
                'attacker_ip': source_ip,
                'tool_used': detected_tool,
                'raw_user_agent': user_agent if user_agent != '' else "[BLANK]",
                'status': action
            })

# ==========================================
# PHASE 3: THE AUTOMATED THREAT REPORT
# ==========================================

# PYTHON: Print a warning header.
print("\n🤖====================================================🤖")
print("🤖=== WARNING: AUTOMATED BOT TRAFFIC DETECTED      ===🤖")
print("🤖====================================================🤖\n")

# PYTHON: Check if our bot folder is empty.
if len(bot_alerts) == 0:
    print("STATUS: SECURE. All web traffic appears to be from standard human web browsers.")

# PYTHON: If we caught bots...
else:
    print(f"ALERT: {len(bot_alerts)} instances of automated script traffic detected!\n")
    
    # PYTHON: Loop through the alerts and print exactly what tool they used.
    for alert in bot_alerts:
        print(f"-> [BOT IP]: {alert['attacker_ip']}")
        print(f"   Time: {alert['timestamp']}")
        
        # PYTHON: Let the team know if the bot was blocked or if it successfully connected.
        print(f"   Action Result: {alert['status'].upper()}")
        print(f"   Tool Identified: '{alert['tool_used']}'")
        print(f"   Full ID Badge: {alert['raw_user_agent']}")
        print("   Action Required: Verify if this is an approved internal IT script. If not, block the IP.\n")


🤖====================================================🤖
🤖=== WARNING: AUTOMATED BOT TRAFFIC DETECTED      ===🤖
🤖====================================================🤖

ALERT: 8 instances of automated script traffic detected!

-> [BOT IP]: 198.51.100.78
   Time: 2024-02-03 09:05:30
   Action Result: BLOCKED
   Tool Identified: 'Blank/Missing User-Agent'
   Full ID Badge: [BLANK]
   Action Required: Verify if this is an approved internal IT script. If not, block the IP.

-> [BOT IP]: 198.51.100.78
   Time: 2024-02-03 09:06:15
   Action Result: BLOCKED
   Tool Identified: 'Blank/Missing User-Agent'
   Full ID Badge: [BLANK]
   Action Required: Verify if this is an approved internal IT script. If not, block the IP.

-> [BOT IP]: 91.198.174.192
   Time: 2024-02-03 09:45:30
   Action Result: BLOCKED
   Tool Identified: 'Blank/Missing User-Agent'
   Full ID Badge: [BLANK]
   Action Required: Verify if this is an approved internal IT script. If not, block the IP.

-> [BOT IP]: 45.142.212.61
   

In [11]:
# PYTHON: Import the CSV module to read our structured firewall evidence.
import csv

# PYTHON: Store the location of our evidence file.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING UP THE SCOREBOARD
# ==========================================

# PYTHON: Create an empty dictionary. 
# SOC: This is our scoreboard. We will map an IP address to the total amount of data (in bytes) it has sent and received.
bandwidth_tracker = {}

# ==========================================
# PHASE 2: TALLYING THE BANDWIDTH
# ==========================================

# PYTHON: Open the file securely in read-only mode ('r').
with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through every log entry.
    for row in reader:
        
        # PYTHON: Extract the Source IP (the computer making the connection).
        source_ip = row['source_ip']
        
        # --------------------------------------
        # THE TRAP: COLLECTING THE BYTES
        # --------------------------------------
        
        # PYTHON: Extract 'bytes_in' and 'bytes_out'. Use .get() to default to '0' if the column is entirely missing from the log.
        # SOC: We need to count every single drop of data.
        raw_bytes_in = row.get('bytes_in', '0')
        raw_bytes_out = row.get('bytes_out', '0')
        
        # PYTHON: Safety check. If the log has the column, but the cell is completely blank, force it to be '0'.
        if raw_bytes_in == '':
            raw_bytes_in = '0'
        if raw_bytes_out == '':
            raw_bytes_out = '0'
            
        # PYTHON: Convert the text strings into actual mathematical integers, then add them together.
        # SOC: We want the TOTAL bandwidth footprint (Download + Upload).
        total_session_bytes = int(raw_bytes_in) + int(raw_bytes_out)
        
        # PYTHON: Add this session's bytes to the IP's total score on our scoreboard.
        # SOC: If '192.168.1.10' already has 500 bytes, and this log shows 200 more, their new score is 700.
        bandwidth_tracker[source_ip] = bandwidth_tracker.get(source_ip, 0) + total_session_bytes

# ==========================================
# PHASE 3: SORTING THE SCOREBOARD
# ==========================================

# PYTHON: We need to sort our dictionary. Python's built-in 'sorted()' function helps here.
# We pass it our dictionary items. 
# 'key=lambda item: item[1]' tells Python to sort by the VALUE (the bytes), not the KEY (the IP). 
# 'reverse=True' means we want it sorted from Highest to Lowest (Descending order).
# SOC: We are bringing the biggest bandwidth hogs right to the top of the list.
sorted_talkers = sorted(bandwidth_tracker.items(), key=lambda item: item[1], reverse=True)

# PYTHON: Slice the list to grab only the Top 5. (The index [:5] means "Start at the beginning, stop at the 5th item").
# SOC: We don't need to see all 10,000 devices. We only care about the top offenders.
top_5_talkers = sorted_talkers[:5]

# ==========================================
# PHASE 4: THE NETWORK UTILIZATION REPORT
# ==========================================

# PYTHON: Print the report header.
print("\n📊====================================================📊")
print("📊=== TRAFFIC ANALYSIS: TOP 5 NETWORK TALKERS      ===📊")
print("📊====================================================📊\n")

print("The following IP addresses consumed the most network bandwidth:\n")

# PYTHON: Start a counter so we can rank them 1, 2, 3, 4, 5.
rank = 1

# PYTHON: Loop through our Top 5 list. 'talker' contains both the IP (talker[0]) and the Bytes (talker[1]).
for talker in top_5_talkers:
    
    # PYTHON: Extract the data for readability.
    ip_address = talker[0]
    total_bytes = talker[1]
    
    # PYTHON: Do some quick math to convert raw bytes into Megabytes (MB) so humans can understand it.
    megabytes = round(total_bytes / (1024 * 1024), 2)
    
    # PYTHON: Print the ranked result.
    # SOC: The Incident Response team will look at this list. If #1 is the Database server, that's normal. If #1 is the receptionist's iPad, we have a malware infection.
    print(f"#{rank} - IP: {ip_address}")
    print(f"      Total Data Transferred: {megabytes} MB ({total_bytes} raw bytes)\n")
    
    # PYTHON: Increase the rank counter for the next loop.
    rank += 1


📊====================================================📊
📊=== TRAFFIC ANALYSIS: TOP 5 NETWORK TALKERS      ===📊
📊====================================================📊

The following IP addresses consumed the most network bandwidth:

#1 - IP: 203.0.113.45
      Total Data Transferred: 10.01 MB (10497024 raw bytes)

#2 - IP: 192.168.1.105
      Total Data Transferred: 9.08 MB (9521152 raw bytes)

#3 - IP: 91.198.174.192
      Total Data Transferred: 1.0 MB (1050112 raw bytes)

#4 - IP: 192.168.1.108
      Total Data Transferred: 0.75 MB (783360 raw bytes)

#5 - IP: 192.168.1.125
      Total Data Transferred: 0.01 MB (14336 raw bytes)



In [12]:
# PYTHON: Import our trusty CSV reader for the final time.
import csv

# PYTHON: Store the location of our firewall logs.
file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: SETTING THE WAF (WEB APP FIREWALL) LIMITS
# ==========================================

# PYTHON: Define our threshold integer.
# SOC: A normal user might click a broken link and get a 404 error once or twice. But if one person generates 20 errors in a row, they are attacking the website.
ERROR_SPIKE_THRESHOLD = 20

# PYTHON: Create our tracking whiteboard.
# SOC: We will track both 4xx (Scanning) and 5xx (Crashing) errors for each IP.
# Example: '192.168.1.5' -> {'4xx': 15, '5xx': 0}
error_tracker = {}

# PYTHON: Create the folder for our final incident reports.
web_attack_alerts = []

# ==========================================
# PHASE 2: MONITORING THE WEB TRAFFIC
# ==========================================

# PYTHON: Open the evidence file.
with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    
    # PYTHON: Loop through the timeline.
    for row in reader:
        
        # PYTHON: Extract the data.
        source_ip = row['source_ip']
        port = row['port']
        raw_status = row['status_code']
        
        # --------------------------------------
        # THE TRAP: COUNTING THE ERRORS
        # --------------------------------------
        
        # PYTHON: Rule 1 - Is this actually web traffic?
        # SOC: Web traffic happens on Port 80 (HTTP) or Port 443 (HTTPS).
        if port == '80' or port == '443':
            
            # PYTHON: Rule 2 - Does the log actually have a status code?
            if raw_status != '':
                
                # PYTHON: Convert the status code into a math number.
                status_code = int(raw_status)
                
                # PYTHON: Initialize the IP on our whiteboard if they aren't there yet.
                if source_ip not in error_tracker:
                    error_tracker[source_ip] = {'4xx_count': 0, '5xx_count': 0}
                
                # PYTHON: Check if it's a 400-level error (Client Error / Scanning).
                if 400 <= status_code <= 499:
                    error_tracker[source_ip]['4xx_count'] += 1
                    
                # PYTHON: Check if it's a 500-level error (Server Error / Crashing).
                elif 500 <= status_code <= 599:
                    error_tracker[source_ip]['5xx_count'] += 1

# ==========================================
# PHASE 3: EVALUATING THE ATTACKS
# ==========================================

# PYTHON: The file reading is complete. Now we review the final counts.
for ip, counts in error_tracker.items():
    
    # PYTHON: Calculate the TOTAL number of errors this IP caused.
    total_errors = counts['4xx_count'] + counts['5xx_count']
    
    # PYTHON: Did they cross our red-line threshold?
    if total_errors >= ERROR_SPIKE_THRESHOLD:
        
        # PYTHON: Determine the likely attack type based on the math.
        # SOC: If they have more 5xx errors, they are trying to break the server. Otherwise, they are just scanning for hidden files.
        if counts['5xx_count'] > counts['4xx_count']:
            attack_type = "Server Crash Attempt / Injection (Heavy 5xx)"
        else:
            attack_type = "Hidden Directory Brute-Forcing (Heavy 4xx)"
            
        # PYTHON: Save the completed alert.
        web_attack_alerts.append({
            'attacker_ip': ip,
            'total_errors': total_errors,
            '4xx': counts['4xx_count'],
            '5xx': counts['5xx_count'],
            'diagnosis': attack_type
        })

# ==========================================
# PHASE 4: THE WAF INCIDENT REPORT
# ==========================================

# PYTHON: Print the final threat report header.
print("\n🌐====================================================🌐")
print("🌐=== CRITICAL: WEB APPLICATION ATTACK DETECTED    ===🌐")
print("🌐====================================================🌐\n")

if len(web_attack_alerts) == 0:
    print("STATUS: SECURE. Web servers are operating normally with standard error rates.")
else:
    print(f"URGENT: {len(web_attack_alerts)} IP addresses are actively attacking the web servers!\n")
    
    for alert in web_attack_alerts:
        print(f"-> [HOSTILE IP]: {alert['attacker_ip']}")
        print(f"-> [THREAT DIAGNOSIS]: {alert['diagnosis']}")
        print(f"   Total Errors Generated: {alert['total_errors']}")
        print(f"   Breakdown: {alert['4xx']} Client Errors (4xx) | {alert['5xx']} Server Errors (5xx)")
        print("-> [ACTION REQUIRED]: Implement an immediate IP ban on the Web Application Firewall (WAF).\n")


🌐====================================================🌐
🌐=== CRITICAL: WEB APPLICATION ATTACK DETECTED    ===🌐
🌐====================================================🌐

STATUS: SECURE. Web servers are operating normally with standard error rates.


In [13]:
# PYTHON: Import our tools. We need 'csv' to read the logs, and 'datetime' to calculate exact time gaps.
import csv
from datetime import datetime

file_path = 'Siem_analysis_enterprise.csv'

# ==========================================
# PHASE 1: TRACKING THE HEARTBEATS
# ==========================================
# SOC: We need to map every internal computer to the external IPs it talks to, AND record the exact time of every connection.
# Example: {'192.168.1.10': {'8.8.8.8': [Time1, Time2, Time3]}}
connection_history = {}

with open(file_path, mode='r') as file:
    reader = csv.DictReader(file)
    for row in reader:
        source_ip = row['source_ip']
        dest_ip = row['destination_ip']
        
        # PYTHON: Skip internal traffic. We only care about computers talking to the outside internet.
        # SOC: Internal IPs usually start with '10.' or '192.168.'.
        if dest_ip.startswith('10.') or dest_ip.startswith('192.168.'):
            continue
            
        try:
            time_obj = datetime.strptime(row['timestamp'], '%Y-%m-%d %H:%M:%S')
        except ValueError:
            continue
            
        # PYTHON: Build our nested dictionary (A dictionary inside a dictionary!)
        if source_ip not in connection_history:
            connection_history[source_ip] = {}
        if dest_ip not in connection_history[source_ip]:
            connection_history[source_ip][dest_ip] = []
            
        # PYTHON: Save the timestamp of this connection.
        connection_history[source_ip][dest_ip].append(time_obj)

# ==========================================
# PHASE 2: CALCULATING ROBOTIC CONSISTENCY
# ==========================================
beacon_alerts = []

# PYTHON: Loop through every computer, and every external server it talked to.
for source, destinations in connection_history.items():
    for dest, timestamps in destinations.items():
        
        # SOC: We only analyze connections that happened at least 4 times. 1 or 2 connections isn't a heartbeat.
        if len(timestamps) >= 4:
            
            # PYTHON: Calculate the time gaps (deltas) between each connection.
            time_gaps = []
            for i in range(1, len(timestamps)):
                gap = (timestamps[i] - timestamps[i-1]).total_seconds()
                time_gaps.append(gap)
                
            # PYTHON: Find the max gap and min gap.
            max_gap = max(time_gaps)
            min_gap = min(time_gaps)
            
            # SOC: The "Jitter" Calculation. 
            # If the longest gap was 3605 seconds, and the shortest was 3598 seconds, the difference is only 7 seconds. 
            # Humans are never that perfectly consistent. Only malware is.
            variance = max_gap - min_gap
            
            # TRIGGER: If the variance between connections is less than 10 seconds, it's a robot!
            if variance < 10:
                beacon_alerts.append({
                    'infected_machine': source,
                    'c2_server': dest,
                    'average_interval': sum(time_gaps) / len(time_gaps)
                })

# Print the Alerts
print("\n📡 === CRITICAL: C2 BEACONING DETECTED === 📡")
for alert in beacon_alerts:
    print(f"-> Host {alert['infected_machine']} is phoning home to Hacker Server {alert['c2_server']}!")
    print(f"-> Heartbeat Interval: Exactly every {round(alert['average_interval'])} seconds.\n")


📡 === CRITICAL: C2 BEACONING DETECTED === 📡


In [15]:
# PYTHON: The requests library allows Python to talk to the internet.
import requests
import json

# SOC: The suspicious IP we caught in Task 4 (Brute Force).
suspicious_ip = "185.220.101.33"

# PYTHON: We define the URL of the Threat Intelligence Database.
api_url = f"https://api.abuseipdb.com/api/v2/check"

# SOC: Your secret badge ID that proves you are a registered security analyst.
headers = {
    'Accept': 'application/json',
    'Key': 'YOUR_SECRET_API_KEY_HERE'
}

# PYTHON: We package our query (the IP) and send it over the internet.
querystring = {'ipAddress': suspicious_ip, 'maxAgeInDays': '90'}
response = requests.get(url=api_url, headers=headers, params=querystring)

# PYTHON: The database replies with a JSON package. We decode it.
intel_report = response.json()

# SOC: Now we extract the global intelligence!
is_malicious = intel_report['data']['abuseConfidenceScore']
isp = intel_report['data']['isp']
domain = intel_report['data']['domain']

print(f"--- THREAT INTEL REPORT FOR {suspicious_ip} ---")
print(f"Malicious Confidence Score: {is_malicious}%")
print(f"Internet Provider: {isp} (Associated Domain: {domain})")

# If the score is over 80%, the global community agrees this is a hacker.

ConnectionError: HTTPSConnectionPool(host='api.abuseipdb.com', port=443): Max retries exceeded with url: /api/v2/check?ipAddress=185.220.101.33&maxAgeInDays=90 (Caused by NameResolutionError("HTTPSConnection(host='api.abuseipdb.com', port=443): Failed to resolve 'api.abuseipdb.com' ([Errno 11001] getaddrinfo failed)"))

In [16]:
import requests

def isolate_machine(infected_ip):
    """
    This function sends an API command directly to CrowdStrike or Microsoft Defender 
    to physically disconnect a laptop from the network.
    """
    print(f"[SOAR AUTOMATION] Isolating {infected_ip} from the network to prevent Ransomware spread...")
    
    # The API endpoint for our endpoint security software
    firewall_api = "https://api.our-security-tool.com/v1/isolate"
    
    # The payload commanding the system to lock down the IP
    payload = {
        "target_ip": infected_ip,
        "action": "network_containment",
        "reason": "Automated Response: Ransomware / C2 Beaconing Detected"
    }
    
    # We send a POST request (a command), rather than a GET request (a question).
    # requests.post(firewall_api, json=payload, headers={'Authorization': 'Bearer SECRET_TOKEN'})
    
    print("[SUCCESS] Host isolated. The threat is contained.")

# Imagine our Beaconing script from Concept 1 just found a hit. 
# Instead of just printing it, we immediately pass the infected machine to our SOAR function:
bad_machine = "192.168.1.105"
isolate_machine(bad_machine)

[SOAR AUTOMATION] Isolating 192.168.1.105 from the network to prevent Ransomware spread...
[SUCCESS] Host isolated. The threat is contained.


In [ ]:
# build a password cracker
# a pass word cracker is tool that cracks password, 
# using the hashlib 
# a hashlib module is used to hash an input
# is a method of converting an input into fix length size string usually hexadecimal and cannot be reversed
# a hash cannot be reversed but can be cracked

# we hash to verify authenticity

# the password to be cracked should be hashed, the we create a word list on a notepad or txt
# convert the plain text to hash to mass=tch the target hash
# using a loop 
# run through to match 

# mutate the password to help crack the password better.

In [6]:
with open('wordlist.txt' , 'w') as file:
    file.write('''
            Hello
            password
            123456
            password123
            admin
            letmein
            qwerty
            superman
            monkey
            P@ssword1!
            ''')

In [18]:
import hashlib
def hash_word(word):
    return hashlib.md5(word.encode()).hexdigest()


def crack_password(target_hash, wordlist):
    with open('wordlist.txt', 'r') as file:
        for word in file:
            word = word.strip()
            hashed = hash_word(word)
            
            mutations = mutate(word)
            
            for mutation in mutations:
                hashed = hash_word(mutation)
                if hashed == target_hash:
                    return f'Cracked {mutation}'
            
            
            
        return 'No hash matches the target'
    

def mutate(word):
    mutations = []
    mutations.append(word + "123")
    mutations.append(word.lower())
    mutations.append(word.capitalize())
    mutations.append(word.upper())
    mutations.append(word.replace('a', '@'))
    mutations.append(word.replace('e', '3'))
    mutations.append(word.replace('i', '!'))
    mutations.append(word)
    return mutations
    
    
    
    
target = hash_word("P@ssword")
crack_password(target, 'wordlist.txt')

'No hash matches the target'

In [ ]:
# hashlib, socket and scalp

In [19]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Reproducibility
np.random.seed(42)
random.seed(42)

n = 5000

# Sample geographic hierarchy
countries = ["Nigeria", "Ghana", "Kenya", "South Africa", "Egypt"]
regions = {
    "Nigeria": ["Lagos", "FCT", "Kano", "Rivers", "Oyo"],
    "Ghana": ["Greater Accra", "Ashanti", "Northern"],
    "Kenya": ["Nairobi", "Mombasa", "Kisumu"],
    "South Africa": ["Gauteng", "Western Cape", "KwaZulu-Natal"],
    "Egypt": ["Cairo", "Giza", "Alexandria"]
}

cities = {
    "Lagos": ["Ikeja", "Lekki", "Victoria Island"],
    "FCT": ["Abuja"],
    "Kano": ["Kano"],
    "Rivers": ["Port Harcourt"],
    "Oyo": ["Ibadan"],
    "Greater Accra": ["Accra"],
    "Ashanti": ["Kumasi"],
    "Northern": ["Tamale"],
    "Nairobi": ["Nairobi"],
    "Mombasa": ["Mombasa"],
    "Kisumu": ["Kisumu"],
    "Gauteng": ["Johannesburg", "Pretoria"],
    "Western Cape": ["Cape Town"],
    "KwaZulu-Natal": ["Durban"],
    "Cairo": ["Cairo"],
    "Giza": ["Giza"],
    "Alexandria": ["Alexandria"]
}

branch_types = ["Retail", "Corporate", "Franchise"]
status_options = ["Active", "Closed", "Under Renovation"]

data = []

for i in range(n):
    country = random.choice(countries)
    region = random.choice(regions[country])
    city = random.choice(cities[region])

    # Population logic (realistic ranges)
    city_population = np.random.randint(100_000, 15_000_000)
    regional_population = city_population + np.random.randint(500_000, 20_000_000)

    branch_type = random.choice(branch_types)

    # Opening date (last 15 years)
    start_date = datetime.now() - timedelta(days=15*365)
    opening_date = start_date + timedelta(days=random.randint(0, 15*365))

    operational_status = np.random.choice(status_options, p=[0.8, 0.1, 0.1])

    # Customer count depends on population
    customer_count = int(city_population * np.random.uniform(0.001, 0.02))

    # Sales volume proportional to customers
    sales_volume = int(customer_count * np.random.uniform(0.8, 1.5))

    # Average transaction value varies by branch type
    if branch_type == "Corporate":
        avg_transaction = np.random.uniform(200, 1000)
    elif branch_type == "Franchise":
        avg_transaction = np.random.uniform(50, 300)
    else:
        avg_transaction = np.random.uniform(20, 150)

    revenue = sales_volume * avg_transaction

    # Cost ratio
    cost = revenue * np.random.uniform(0.6, 0.9)

    profit = revenue - cost
    profit_margin = (profit / revenue) * 100 if revenue > 0 else 0

    # Growth rate influenced by status
    if operational_status == "Active":
        growth_rate = np.random.uniform(2, 25)
    elif operational_status == "Under Renovation":
        growth_rate = np.random.uniform(-10, 5)
    else:
        growth_rate = np.random.uniform(-20, 2)

    row = [
        i + 1,
        f"BU_{random.randint(1,10)}",
        f"BR_{random.randint(1000,9999)}",
        f"{city} Branch",
        country,
        region,
        city,
        city_population,
        regional_population,
        branch_type,
        opening_date.date(),
        operational_status,
        round(revenue, 2),
        round(cost, 2),
        round(profit, 2),
        round(profit_margin, 2),
        customer_count,
        sales_volume,
        round(avg_transaction, 2),
        round(growth_rate, 2)
    ]

    data.append(row)

columns = [
    "Record_ID",
    "Business_Unit",
    "Branch_ID",
    "Branch_Name",
    "Country",
    "State_Region",
    "City",
    "City_Population",
    "Regional_Population",
    "Branch_Type",
    "Opening_Date",
    "Operational_Status",
    "Revenue",
    "Cost",
    "Profit",
    "Profit_Margin (%)",
    "Customer_Count",
    "Sales_Volume",
    "Average_Transaction_Value",
    "Growth_Rate (%)"
]

df = pd.DataFrame(data, columns=columns)

# Save to file
df.to_csv("bi_dataset_5000_rows.csv", index=False)

print("Dataset generated successfully!")
print(df.head())

Dataset generated successfully!
   Record_ID Business_Unit Branch_ID             Branch_Name  Country  \
0          1          BU_4   BR_3286  Victoria Island Branch  Nigeria   
1          2          BU_1   BR_1488           Ibadan Branch  Nigeria   
2          3          BU_1   BR_4257            Abuja Branch  Nigeria   
3          4          BU_5   BR_1106             Giza Branch    Egypt   
4          5          BU_3   BR_4527           Tamale Branch    Ghana   

  State_Region             City  City_Population  Regional_Population  \
0        Lagos  Victoria Island         14902022             31496500   
1          Oyo           Ibadan         14955267             25083786   
2          FCT            Abuja          1362752             11647385   
3         Giza             Giza         13331055             26479692   
4     Northern           Tamale          2763046             14222060   

  Branch_Type Opening_Date Operational_Status       Revenue          Cost  \
0   Corporate